# Regression models
## General research

The regression models are part of the supervised machine learning.

### Linear regression
The linear regression is a model that tries to predict a target variable using one or multiple independant(s) value(s).
Its formula is : y = α0 + α1*x + ε
The training consists to find the α that find the best predicted y.
In order to find the most optimal data you need to evaluate the models, one of the most used ways is the RMSE (stands for Root Mean Square Error) which uses the differences between the predicted value and the real value to evaluate the output.

![Linear regression](data/images/imagelinear.png)

#### Multiple linear regression
The Multiple linear regression is a model similar to the linear regression that instead uses multiple factors. Its formula is : y = α0 + α1*x1 + α2*x2 + αn*xn + ε
Its main advantage is the weight you can give to one variable without affecting the weights of the other in the equation.


![Multiple linear regression](data/images/imagemultiple.png)

##### Polynomial regression
The polynomial regression can be counted as a *sub-model* of the multiple linear regression. It can give a curved shape to the prediction while staying a linear model. Its formula is : y = α0 + α1*x + α2*x² + αn*xⁿ + ε
The model can use a power in its formula which can weight other features.
Its main advantage is its non-linear flexibility.
Its main risk is overfitting: the higher the degree, the more the curve follows the noise instead of the trend.

![Polynomial regression](data/images/imagepolynomial.png)

### Poisson regression
The poisson regression is a model that specializes in counts. Like defects/batch, rates of accident, sales/shop.
Its main characteristic is that the mean is related to the distribution. It's better in low/close to zero counts.
You will have a distribution chance, like, "what is the chance that y = 6?"

![Poisson regression](data/images/imagepoisson.png)

### Support vector regression
The Support vector regression is a model that looks like the linear regression in its formula : ŷ = w·x + b
But there is a new thing to take in account, a margin for error directly integrated in the model. It's called ε-tube and enables to better process outliers. It's not going to penalize the data in this margin and the model will train on slightly false predictions but will be able to process broader data.

![Support vector regression](data/images/imagesvr.png)

### Gradient boosting
The gradient boosting model is actually a technique that combines multiple models into one in order to fight overfitting and improve accuracy.
It uses the gradient descent and boosting method. Boosting is where the collective is iteratively improved with each model. Bagging is where the collective is imroved with models working in tandem.
We're focused on boosting and more specificaly XGBoost which enables the parallel learning using multiple cores in the CPU during model training.

![Gradient boosting](data/images/imagegradient.png)

### Random forest
The random forest model is actually an ensemble technique as well. It uses multiple decision trees during its training period and then makes an average of all the tree's results. It is part of the bagging techniques as its models are working independantly from the results of other trees.
Not all features are used in the trees, they are randomly picked and used for training by some of trees.
The data is going through nodes with multiple trees at the same time and the one making the best separation is chosen.
This process times 100 or 1000 makes it more robust and accurate.

![Random forest](data/images/imagerandomforest.png)

### Elastic net
The Elastic net model is a linear regression with a bonus. It not only works to minimize the errors (linear regression), it also minimizes the impact of coefficients. It is the combinaison between Ridge and Lasso. (Lasso) deletes useless variables, (Ridge) handles correlated groups.
It asks to encode the features, as other models might.
But it is easy to implement as its tuning is done automaticaly.

![Elastic net](data/images/imageelastic.png)

## The chosen ones
In our case, the regression models we chose are the last 3:
- Support Vector Regression (SVR)
- Gradient Boosting (or just Boosting)
- Random Forest

These are the most robust and "real-estate-oriented".
They can easily treat the natural skewness in our dataset and are within the limits of size of our dataset as well.
We'll see how much of tuning we'll need, after training and evalutation.

# Modeling and grid search

In this part, we train the 3 chosen models (SVR, Random Forest, Gradient Boosting), we tune their settings with a **grid search**, then we compare them.

The plan:

1. **Prepare the data**, with the same train / test split for every model.
2. **Measure a starting point**: each model with its default settings.
3. **Grid search**: try many settings for each model and keep the best ones.
4. **Final exam**: test all the models once on the test set and choose the winner.

## 1. Data and train / test split

- We load the cleaned file and keep the **7 features** chosen in `notebook.ipynb`: `grade`, `sqm_living`, `sqm_lot`, `waterfront`, `view`, `lat`, `long`.
- The target is `log(price)`, as decided in the cleaning step (it reduces the weight of the luxury houses).
- We use **exactly the same split** as in `notebook.ipynb` (`test_size=0.2`, `random_state=42`). The `random_state` fixes the "random" draw: every model is trained and tested on the same houses. Without it, each model would get different houses, and one could look better just because it got easier houses to predict.
- The **test set** (20% of the houses) is put aside and not used until the last step. It is our "final exam": if we looked at it to make our choices, the final score would be too optimistic.

In [6]:
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cleaned data (made in notebook.ipynb)
df = pd.read_csv("data/clean/kc_house_data_clean.csv")

# The 7 features chosen in the feature selection
final_features = ["grade", "sqm_living", "sqm_lot", "waterfront", "view", "lat", "long"]
X = df[final_features]

# Target: log of the price
y = np.log(df["price"])

# Same split as in notebook.ipynb: random_state=42 gives the same houses to every model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train set:", X_train.shape)
print("Test set :", X_test.shape)

Train set: (17284, 7)
Test set : (4322, 7)


## 2. Starting point: the models with their default settings

Every model has **hyperparameters**: settings that we choose before the training (for example the number of trees of a Random Forest). The model does not learn them by itself. Here we first use the default values of scikit-learn, to have a reference score. Then we will see if the grid search does better.

**How do we give a score without touching the test set?** With **cross-validation** (`cv=5`): the train set is cut into 5 parts. The model is trained on 4 parts and scored on the 5th one, and this is repeated 5 times so that each part is used once for scoring. The score of the model is the average of the 5 scores.

```text
Round 1 : [SCORE] [train] [train] [train] [train]
Round 2 : [train] [SCORE] [train] [train] [train]
Round 3 : [train] [train] [SCORE] [train] [train]
Round 4 : [train] [train] [train] [SCORE] [train]
Round 5 : [train] [train] [train] [train] [SCORE]
                                   -> average of the 5 scores
```

**The score used is the MAE** (mean absolute error). It is the simplest one to explain: "on average, the prediction is wrong by X". As the target is `log(price)`, a MAE of 0.13 means that the prediction is **about 13% away** from the real price, on average. scikit-learn always looks for the highest score, so it gives the MAE with a minus sign (`neg_mean_absolute_error`): we put the minus sign back to read it.

The SVR is put in a `Pipeline` with a `StandardScaler`: the SVR measures distances between houses, so all the features must be on the same scale (otherwise `sqm_lot`, in thousands of m², would hide `grade`, from 1 to 13). The trees (Random Forest, Gradient Boosting) do not need it.

In [7]:
# The 3 models with their default settings
# random_state=42 makes the trees give the same result each time we run the notebook
default_models = {
    "SVR": Pipeline([
        ("scaler", StandardScaler()),  # puts every feature on the same scale
        ("svr", SVR()),
    ]),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

# Cross-validation score (MAE on the log of the price) of each model, on the train set only
default_cv_mae = {}
for name, model in default_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5,     # 5 cuts in dataset to train
                             scoring="neg_mean_absolute_error", n_jobs=-1)
    default_cv_mae[name] = -scores.mean()  # minus sign: back to a positive error
    print(name, ": MAE =", round(default_cv_mae[name], 4))

SVR : MAE = 0.1484
Random Forest : MAE = 0.1312
Gradient Boosting : MAE = 0.1409


## 3. Grid search

**The idea:** we write a list of values to try for each hyperparameter (the "grid"). The grid search trains the model with **every combination** of these values, gives each combination a cross-validation score (the 5 rounds seen above), and keeps the best combination.

```text
n_estimators : [100, 300, 500]
max_depth    : [10, 20, None]     ->  3 x 3 x 3 = 27 combinations
max_features : [0.33, 0.5, 1.0]       x 5 rounds = 135 trainings
```

The cost is multiplied by the number of values: adding one value to a line can make the search much longer. So we keep **3 hyperparameters per model**, the ones that change the result the most, with values far from each other to cover a wide range.

How we read the result:

- `best_params_`: the best combination.
- `best_score_`: its cross-validation score (with a minus sign).
- `best_estimator_`: the model with the best combination, **already retrained on the whole train set** (`refit=True` by default), ready to predict.

Two small helpers are used for the 3 models:

- `show_top` prints the 5 best combinations, to see if the winner is far ahead or not.
- `check_edges` checks if the best value is at the **edge of the grid** (the smallest or the biggest value tested). If it is, the real best value may be further, outside the grid: we have to test further in that direction.

Note: when the grid is too big, `RandomizedSearchCV` can be used instead. It only tries a given number of combinations taken at random. Our grids are small enough to try every combination.

In [8]:
# Prints the 5 best combinations of a grid search
def show_top(grid, n=5):
    results = pd.DataFrame(grid.cv_results_)
    results["MAE"] = -results["mean_test_score"]  # minus sign: back to a positive error
    results = results.sort_values("MAE")
    print(results[["params", "MAE"]].head(n).to_string(index=False))


# Says, for each hyperparameter, if the best value is at the edge of the grid
def check_edges(grid, param_grid):
    for name, values in param_grid.items():
        best = grid.best_params_[name]
        if best == values[0] or best == values[-1]:
            print(name, "=", best, "-> at the edge of the grid")
        else:
            print(name, "=", best, "-> inside the grid")

### 3.1 SVR

The SVR draws a curve that goes through the houses, with a "tube" around it. The 3 hyperparameters tuned:

| Hyperparameter | What it does | Values tested |
| --- | --- | --- |
| `C` | How strict the model is with the houses outside the tube. Small: a smooth curve that accepts errors. Big: a curve that tries to pass close to every house (risk of learning by heart). | 1, 3, 10, 30 |
| `gamma` | How far the influence of one house goes. Small: each house influences many others (smooth curve). Big: each house only influences its close neighbors (very bendy curve). | 0.05, 0.15, 0.5 |
| `epsilon` | The width of the tube. Errors smaller than this are ignored. 0.1 on the log is about 10% of the price. | 0.05, 0.1 |

Why these values: the default values are `C=1`, `gamma="scale"` (here about 0.14) and `epsilon=0.1`, so the grid goes around them: `gamma` about 3 times smaller and 3 times bigger, `C` from 1 to 30. We stop `C` at 30 because the SVR becomes very slow when `C` grows: one training takes about 5 s with `C=1`, but about 47 s with `C=100`.

In the grid, the names start with `svr__` (two underscores): the SVR is inside the `Pipeline`, at the step called `"svr"`, so scikit-learn needs to know to which step the setting belongs.

4 x 3 x 2 = 24 combinations, x 5 rounds = 120 trainings. This is the slowest model (a few minutes).

In [9]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR()),
])

# The grid: the values to try for each hyperparameter
svr_grid = {
    "svr__C": [1, 3, 10, 30],
    "svr__gamma": [0.05, 0.15, 0.5],
    "svr__epsilon": [0.05, 0.1],
}

# n_jobs=-1: uses all the processor cores, to train several combinations at the same time
svr_search = GridSearchCV(svr_pipeline, svr_grid, cv=5,
                          scoring="neg_mean_absolute_error", n_jobs=-1)

start = time.time()
svr_search.fit(X_train, y_train)  # train set only
print("Time:", round(time.time() - start), "seconds")
print("Best combination:", svr_search.best_params_)
print("MAE:", round(-svr_search.best_score_, 4))
print()
show_top(svr_search)
print()
check_edges(svr_search, svr_grid)

Time: 515 seconds
Best combination: {'svr__C': 3, 'svr__epsilon': 0.05, 'svr__gamma': 0.5}
MAE: 0.1375

                                                 params      MAE
 {'svr__C': 3, 'svr__epsilon': 0.05, 'svr__gamma': 0.5} 0.137524
 {'svr__C': 1, 'svr__epsilon': 0.05, 'svr__gamma': 0.5} 0.137619
  {'svr__C': 3, 'svr__epsilon': 0.1, 'svr__gamma': 0.5} 0.137842
  {'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma': 0.5} 0.138205
{'svr__C': 10, 'svr__epsilon': 0.05, 'svr__gamma': 0.5} 0.139311

svr__C = 3 -> inside the grid
svr__gamma = 0.5 -> at the edge of the grid
svr__epsilon = 0.05 -> at the edge of the grid


### 3.2 Random Forest

The Random Forest makes the average of many decision trees, each one trained on a random part of the data. The 3 hyperparameters tuned:

| Hyperparameter | What it does | Values tested |
| --- | --- | --- |
| `n_estimators` | The number of trees. More trees give a more stable average, but a longer training. | 100, 300, 500 |
| `max_depth` | The maximum number of questions a tree can ask ("grade > 8?", then "lat > 47.6?", ...). Small: simple trees that may miss details. `None`: no limit. | 10, 20, None |
| `max_features` | The share of the features that each split of a tree can look at. 1.0: all of them (default). Smaller: the trees are more different from each other, so their average is better. | 0.33, 0.5, 1.0 |

3 x 3 x 3 = 27 combinations, x 5 rounds = 135 trainings.

In [10]:
rf_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [10, 20, None],
    "max_features": [0.33, 0.5, 1.0],
}

rf_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_grid, cv=5,
                         scoring="neg_mean_absolute_error", n_jobs=-1)

start = time.time()
rf_search.fit(X_train, y_train)
print("Time:", round(time.time() - start), "seconds")
print("Best combination:", rf_search.best_params_)
print("MAE:", round(-rf_search.best_score_, 4))
print()
show_top(rf_search)
print()
check_edges(rf_search, rf_grid)

Time: 797 seconds
Best combination: {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 500}
MAE: 0.1283

                                                       params      MAE
  {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 500} 0.128328
  {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 300} 0.128473
{'max_depth': None, 'max_features': 0.5, 'n_estimators': 500} 0.128620
{'max_depth': None, 'max_features': 0.5, 'n_estimators': 300} 0.128782
 {'max_depth': 20, 'max_features': 0.33, 'n_estimators': 500} 0.128814

n_estimators = 500 -> at the edge of the grid
max_depth = 20 -> inside the grid
max_features = 0.5 -> inside the grid


### 3.3 Gradient Boosting

The Gradient Boosting builds trees one after the other: each new tree corrects the errors of the previous ones. The 3 hyperparameters tuned:

| Hyperparameter | What it does | Values tested |
| --- | --- | --- |
| `n_estimators` | The number of trees (the number of corrections). | 200, 500, 1000 |
| `learning_rate` | The size of each correction. Small: careful small steps (needs more trees). Big: big steps (risk of over-correcting). | 0.03, 0.1, 0.3 |
| `max_depth` | The depth of each tree. Gradient Boosting uses small trees (default: 3). | 3, 5, 7 |

`n_estimators` and `learning_rate` work together: small steps need many trees. This is why they are tuned together, in the same grid.

3 x 3 x 3 = 27 combinations, x 5 rounds = 135 trainings.

In [11]:
gb_grid = {
    "n_estimators": [200, 500, 1000],
    "learning_rate": [0.03, 0.1, 0.3],
    "max_depth": [3, 5, 7],
}

gb_search = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_grid, cv=5,
                         scoring="neg_mean_absolute_error", n_jobs=-1)

start = time.time()
gb_search.fit(X_train, y_train)
print("Time:", round(time.time() - start), "seconds")
print("Best combination:", gb_search.best_params_)
print("MAE:", round(-gb_search.best_score_, 4))
print()
show_top(gb_search)
print()
check_edges(gb_search, gb_grid)

Time: 455 seconds
Best combination: {'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 500}
MAE: 0.1261

                                                       params      MAE
 {'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 500} 0.126115
  {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200} 0.126436
{'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 1000} 0.126437
{'learning_rate': 0.03, 'max_depth': 5, 'n_estimators': 1000} 0.126526
  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500} 0.126709

n_estimators = 500 -> inside the grid
learning_rate = 0.03 -> at the edge of the grid
max_depth = 7 -> at the edge of the grid


## 4. Final exam on the test set

Now, and only now, we use the test set. Each model (default settings, and settings found by the grid search) predicts the price of the 4,322 test houses, which it has never seen. The default models are retrained on the whole train set first.

The predictions are in `log(price)`, so we turn them back into dollars with `exp()` to compute:

- **MAE**: the average error, in dollars.
- **RMSE**: also an average error in dollars, but big errors count more. It is higher than the MAE when the model makes some very big mistakes (often on the luxury houses).
- **R²**: the share of the price variation explained by the model (1 = perfect). It is computed on the log, as in the feature selection.

In [12]:
# Gives the 3 metrics of a trained model on the test set
def evaluate(model):
    y_pred_log = model.predict(X_test)
    y_pred = np.exp(y_pred_log)  # back to dollars
    y_true = np.exp(y_test)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_test, y_pred_log)
    return round(mae), round(rmse), round(r2, 4)


# The models found by the grid search (already retrained on the whole train set)
tuned_models = {
    "SVR": svr_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "Gradient Boosting": gb_search.best_estimator_,
}
tuned_cv_mae = {
    "SVR": -svr_search.best_score_,
    "Random Forest": -rf_search.best_score_,
    "Gradient Boosting": -gb_search.best_score_,
}

rows = []
for name in default_models:
    # Default settings: we train the model on the whole train set, then test it
    default_models[name].fit(X_train, y_train)
    mae, rmse, r2 = evaluate(default_models[name])
    rows.append([name, "default", round(default_cv_mae[name], 4), mae, rmse, r2])

    # Settings found by the grid search
    mae, rmse, r2 = evaluate(tuned_models[name])
    rows.append([name, "grid search", round(tuned_cv_mae[name], 4), mae, rmse, r2])

comparison = pd.DataFrame(rows, columns=["Model", "Settings", "CV MAE (log)", "Test MAE ($)", "Test RMSE ($)", "Test R2"])
comparison

,Model,Settings,CV MAE (log),Test MAE ($),Test RMSE ($),Test R2
0,SVR,default,0.1484,80510,155399,0.8572
1,SVR,grid search,0.1375,76383,175746,0.8639
2,Random Forest,default,0.1312,70357,129835,0.8827
3,Random Forest,grid search,0.1283,69374,128487,0.8871
4,Gradient Boosting,default,0.1409,76852,139662,0.8708
5,Gradient Boosting,grid search,0.1261,68455,124199,0.8920


## 5. Saving the best model

The winner is saved in `models/best_model.joblib`. The Flask application will load this file to predict prices, without having to train the model again. It expects the same 7 features, in the same order, and it predicts `log(price)`: the application will have to apply `exp()` to give a price in dollars.

In [13]:
# The winner is chosen with the cross-validation score (the test set is only used to check it)
best_name = min(tuned_cv_mae, key=tuned_cv_mae.get)
best_model = tuned_models[best_name]
print("Best model:", best_name)

joblib.dump(best_model, "models/best_model.joblib")
print("Saved in models/best_model.joblib")

Best model: Gradient Boosting
Saved in models/best_model.joblib
